# 02 – Globale Landtemperatur aus Stationsdaten

Wir berechnen die globale Landtemperatur selbst aus den ~28.000 Stationen von **GHCN-Monthly v4** (NOAA) und vergleichen sie mit **NASA GISTEMP** (nur Land).

Methode (`src/klima/weltweit.py`, SPEC Abschnitt 3.1):
1. Monatliche **Anomalien** je Station gegenüber 1951–1980 (mind. 20 Referenzjahre je Kalendermonat),
2. **5°×5°-Gitter**: Mittel der Stationsanomalien je Zelle,
3. **flächengewichtetes Mittel** über alle besetzten Zellen (cos der Breite).

Beides wird für die **unbereinigten (QCU)** und die **homogenisierten (QCF)** Daten gerechnet.

In [ ]:
%matplotlib inline
import pandas as pd

from klima import auswertungen
from klima import weltweit as ww
from klima.anomalien import standard_referenzperiode
from klima.grafik import interaktiv, statisch
from klima.trend import linearer_trend

pd.set_option("display.precision", 2)
REFERENZ = standard_referenzperiode()

## Stationsanomalien und Gitter

In [ ]:
anomalien_qcf = ww.stationsanomalien("qcf", REFERENZ)
print(f"Stationen mit ausreichender Referenz: {anomalien_qcf.stations_id.nunique():,}")
gitter_qcf = ww.gitter_mittel(anomalien_qcf, groesse=5.0)
gitter_qcf.tail()

In [ ]:
gitter_qcu = ww.gitter_mittel(ww.stationsanomalien("qcu", REFERENZ), groesse=5.0)
jahre_qcf = ww.jahresanomalien(gitter_qcf).loc[1880:]
jahre_qcu = ww.jahresanomalien(gitter_qcu).loc[1880:]
jahre_qcf.tail()

## Vergleich mit NASA GISTEMP

Das **globale** Mittel über besetzte Zellen ist nordlastig (mehr Land und Stationen im Norden, wo es sich schneller erwärmt). Das **Halbkugelmittel** ((Nord + Süd) / 2, wie bei CRUTEM) gleicht das aus. GISTEMP überträgt Stationswerte zusätzlich bis 1.200 km weit, auch auf küstennahe Meeresflächen.

In [ ]:
gistemp = auswertungen._gistemp_jahre("global", "land").loc[1880:]
vergleich = pd.DataFrame(
    {
        auswertungen.GISTEMP_LAND: gistemp,
        auswertungen.EIGENE_QCF: jahre_qcf["global"],
        auswertungen.EIGENE_QCU: jahre_qcu["global"],
        auswertungen.EIGENE_HALBKUGEL: jahre_qcf["halbkugelmittel"],
    }
)
pd.DataFrame({name: vars(linearer_trend(vergleich[name], von=1951)) for name in vergleich}).T[
    ["steigung_pro_dekade", "unsicherheit_95", "von", "bis"]
]

In [ ]:
interaktiv.zeitreihen(
    vergleich, "Globale Landtemperatur", referenz=REFERENZ, farben=auswertungen.FARBEN_GLOBAL
)

## Halbkugeln

In [ ]:
statisch.zeitreihen(
    jahre_qcf[["nordhalbkugel", "suedhalbkugel"]].rename(
        columns={"nordhalbkugel": "Nordhalbkugel", "suedhalbkugel": "Südhalbkugel"}
    ),
    "Landtemperatur nach Halbkugel",
    referenz=REFERENZ,
);

## Effekt der Homogenisierung

Die Homogenisierung (Pairwise Homogenization Algorithm der NOAA) korrigiert Sprünge durch Stationsverlegungen, Gerätewechsel und geänderte Beobachtungszeiten. Global macht sie die Vergangenheit etwas kälter – der Erwärmungstrend wird dadurch leicht größer.

In [ ]:
(vergleich[auswertungen.EIGENE_QCF] - vergleich[auswertungen.EIGENE_QCU]).loc[
    [1880, 1900, 1950, 2000, 2024]
]

In [ ]:
effekt = ww.homogenisierungseffekt(von=1951)
print(
    f"{len(effekt)} Stationen, Median {effekt.differenz.median():+.3f} °C/Dekade, "
    f"Anteil mit größerem Trend: {(effekt.differenz > 0).mean():.0%}"
)
effekt.groupby("land").differenz.agg(["count", "median"]).query("count >= 30").sort_values("median")

In [ ]:
statisch.histogramm(
    effekt["differenz"], "Trendänderung je Station durch Homogenisierung", "°C/Dekade"
);

In [ ]:
interaktiv.weltkarte_stationen(
    effekt, "differenz", "Trendänderung durch Homogenisierung", "°C/Dekade"
)

## Abdeckung durch Stationen

In [ ]:
abdeckung = ww.abdeckung(gitter_qcf, anomalien_qcf, groesse=5.0).loc[1880:]
statisch.abdeckung(abdeckung, "Stationsabdeckung", "GHCNm homogenisiert, 5°-Zellen");

## Anomalie je Jahrzehnt (Zeitregler)

In [ ]:
jahrzehnte = ww.jahrzehntmittel(gitter_qcf)
interaktiv.gitterkarte_zeitregler(
    jahrzehnte[jahrzehnte.jahrzehnt >= 1880],
    "jahrzehnt",
    "Temperaturanomalie je Jahrzehnt",
    beschriftung=lambda z: f"{z}er",
)